Контрольні питання
1. Чи можете пояснити словами логіку split-apply-combine, яку виконує groupby() — що саме відбувається на кожному з трьох кроків?
groupby() працює за принципом split-apply-combine.

Split (розділення) — дані розділяються на окремі групи за певною ознакою. Наприклад, якщо використати:

climate.groupby("рік")

то всі дані будуть розділені на чотири групи: 2021, 2022, 2023 та 2024 роки.

Apply (застосування) — до кожної групи застосовується певна операція або агрегувальна функція. Наприклад:

.mean()

обчислює середню температуру для кожного року, а .min() і .max() знаходять мінімальне та максимальне значення.

Combine (об'єднання) — результати для всіх груп об'єднуються в одну підсумкову таблицю.

Отже, groupby() дозволяє розділити дані на групи, виконати над кожною групою потрібні обчислення та отримати спільний результат.

2. Чи розумієте принципову відмінність між pivot() і pivot_table() — чому один завершується помилкою на дублікатах комбінацій індекс/стовпець, а другий — ні?
pivot() використовується для зміни структури таблиці, але він вимагає, щоб кожна комбінація значень index і columns відповідала лише одному значенню.

Наприклад, у моєму наборі даних кожна комбінація "місяць" і "рік" зустрічається один раз, тому:

climate.pivot(
    index="місяць",
    columns="рік",
    values="температура"
)

працює без помилки.

Якщо для однієї комбінації "місяць" та "рік" буде декілька значень, pivot() не знатиме, яке з них потрібно помістити в клітинку, тому виникне помилка про дублікати.

pivot_table() відрізняється тим, що вміє працювати з такими дублікованими значеннями. Вона може об'єднати їх за допомогою агрегувальної функції, наприклад:

aggfunc="mean"

Тобто якщо для одного місяця і року є декілька температур, pivot_table() може порахувати їх середнє значення.

3. Чи можете пояснити, що показує crosstab() і чим він відрізняється від еквівалентного результату через groupby(...).size()?
crosstab() використовується для підрахунку кількості спостережень у різних комбінаціях категорій.

У моїй роботі:

pd.crosstab(
    climate["сезон"],
    climate["тепліше_за_середнє"]
)

показує, скільки спостережень припадає на кожну комбінацію сезону та значення True/False.

Наприклад, можна побачити, скільки зимових, весняних, літніх та осінніх спостережень були теплішими за середньорічну температуру.

Схожий результат можна отримати за допомогою:

climate.groupby(
    ["сезон", "тепліше_за_середнє"]
).size()

Основна відмінність полягає у формі результату. groupby().size() повертає групи та їх кількість, зазвичай у вигляді Series з MultiIndex. crosstab() одразу формує зручну двовимірну таблицю частот, де одна категорія знаходиться в рядках, а інша — у стовпцях.

4. Чи можете пояснити, чому дані для groupby і pivot_table мають бути в довгому (tidy) форматі, а не в таблиці з роками чи місяцями, розтягнутими в назви стовпців?
Довгий, або tidy, формат є зручним для аналізу, тому що кожна змінна знаходиться в окремому стовпці, а кожен рядок відповідає одному спостереженню.

У моєму наборі даних:

місто | рік | місяць | температура

рік, місяць і температура є окремими змінними. Тому pandas легко може виконати:

climate.groupby("рік")

або:

climate.groupby("місяць")

і побудувати:

climate.pivot_table(
    index="місяць",
    columns="рік"
)

Якщо ж роки або місяці записані безпосередньо в назвах стовпців, наприклад 2021_січень, 2021_лютий, 2022_січень, то структура даних стає менш зручною для групування та аналізу. У такому випадку спочатку потрібно перетворити дані у довгий формат, наприклад за допомогою melt().

In [24]:
climate.groupby(["місяць", "рік"]).size()


місяць  рік 
1       2021    1
        2022    1
        2023    1
        2024    1
2       2021    1
        2022    1
        2023    1
        2024    1
3       2021    1
        2022    1
        2023    1
        2024    1
4       2021    1
        2022    1
        2023    1
        2024    1
5       2021    1
        2022    1
        2023    1
        2024    1
6       2021    1
        2022    1
        2023    1
        2024    1
7       2021    1
        2022    1
        2023    1
        2024    1
8       2021    1
        2022    1
        2023    1
        2024    1
9       2021    1
        2022    1
        2023    1
        2024    1
10      2021    1
        2022    1
        2023    1
        2024    1
11      2021    1
        2022    1
        2023    1
        2024    1
12      2021    1
        2022    1
        2023    1
        2024    1
dtype: int64

In [23]:
temperature_pivot_simple = climate.pivot(
    index="місяць",
    columns="рік",
    values="температура"
)

temperature_pivot_simple


рік,2021,2022,2023,2024
місяць,,,,
1,-3.0,-3.3,-4.0,-3.3
2,-2.0,-3.8,-1.8,-3.9
3,3.1,0.8,1.3,1.2
4,10.0,7.9,8.9,8.7
5,14.3,13.5,13.9,15.2
6,18.7,19.2,18.6,19.1
7,22.1,19.6,19.9,20.4
8,19.7,17.5,20.7,18.6
9,14.0,16.0,14.5,13.0


In [21]:
climate[
    (climate["сезон"] == "літо") &
    (climate["тепліше_за_середнє"] == False)
]


,місто,рік,місяць,температура,сезон,тепліше_за_середнє


In [20]:
pd.crosstab(
    climate["сезон"],
    climate["тепліше_за_середнє"]
)


тепліше_за_середнє,False,True
сезон,,
весна,7,5
зима,12,0
літо,0,12
осінь,8,4


In [19]:
climate


,місто,рік,місяць,температура,сезон,тепліше_за_середнє
0,Львів,2021,1,-3.0,зима,False
1,Львів,2021,2,-2.0,зима,False
2,Львів,2021,3,3.1,весна,False
3,Львів,2021,4,10.0,весна,True
4,Львів,2021,5,14.3,весна,True
5,Львів,2021,6,18.7,літо,True
6,Львів,2021,7,22.1,літо,True
7,Львів,2021,8,19.7,літо,True
8,Львів,2021,9,14.0,осінь,True
9,Львів,2021,10,9.0,осінь,False


In [18]:
climate["тепліше_за_середнє"] = climate["температура"] > 9.5


In [17]:
climate["тепліше_за_середнє"] = climate["температура"] > 8.5


In [16]:
climate.head(15)


,місто,рік,місяць,температура,сезон
0,Львів,2021,1,-3.0,зима
1,Львів,2021,2,-2.0,зима
2,Львів,2021,3,3.1,весна
3,Львів,2021,4,10.0,весна
4,Львів,2021,5,14.3,весна
5,Львів,2021,6,18.7,літо
6,Львів,2021,7,22.1,літо
7,Львів,2021,8,19.7,літо
8,Львів,2021,9,14.0,осінь
9,Львів,2021,10,9.0,осінь


In [15]:
climate["сезон"] = climate["місяць"].apply(get_season)


In [14]:
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"


In [13]:
temperature_pivot = climate.pivot_table(
    index="місяць",
    columns="рік",
    values="температура",
    aggfunc="mean"
)

temperature_pivot


рік,2021,2022,2023,2024
місяць,,,,
1,-3.0,-3.3,-4.0,-3.3
2,-2.0,-3.8,-1.8,-3.9
3,3.1,0.8,1.3,1.2
4,10.0,7.9,8.9,8.7
5,14.3,13.5,13.9,15.2
6,18.7,19.2,18.6,19.1
7,22.1,19.6,19.9,20.4
8,19.7,17.5,20.7,18.6
9,14.0,16.0,14.5,13.0


In [12]:
print("Місяць із найбільшим розкидом:", month_stats["std"].idxmax())
print("Найбільше стандартне відхилення:", month_stats["std"].max())


Місяць із найбільшим розкидом: 8
Найбільше стандартне відхилення: 1.3817259737975056


In [11]:
month_stats["std"].idxmax()


np.int64(8)

In [10]:
month_stats = climate.groupby("місяць")["температура"].agg(
    ["mean", "std"]
)

month_stats


,mean,std
місяць,,
1,-3.400,0.424264
2,-2.875,1.129528
3,1.600,1.023067
4,8.875,0.865544
5,14.225,0.727438
6,18.900,0.294392
7,20.500,1.116542
8,19.125,1.381726
9,14.375,1.250000


### Висновок до Завдання 1

За отриманими значеннями середньої температури можна оцінити зміни температури протягом 2021–2024 років. Якщо середня температура поступово зростає, можна говорити про слабку тенденцію до потепління. Якщо значення коливаються без чіткої тенденції, то зміни більше схожі на випадкові коливання. Оскільки аналізуються лише чотири роки, робити висновок про довготривалий кліматичний тренд не можна.


In [9]:
year_stats = climate.groupby("рік")["температура"].agg(
    ["mean", "min", "max"]
)

year_stats


,mean,min,max
рік,,,
2021,8.791667,-3.0,22.1
2022,7.916667,-3.8,19.6
2023,8.300000,-4.0,20.7
2024,8.166667,-3.9,20.4


In [8]:
climate.groupby("рік")["температура"].agg(["mean", "min", "max"])


,mean,min,max
рік,,,
2021,8.791667,-3.0,22.1
2022,7.916667,-3.8,19.6
2023,8.300000,-4.0,20.7
2024,8.166667,-3.9,20.4


In [7]:
climate.groupby("рік")


In [6]:
climate.info()


<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   місто        48 non-null     str    
 1   рік          48 non-null     int64  
 2   місяць       48 non-null     int64  
 3   температура  48 non-null     float64
dtypes: float64(1), int64(2), str(1)
memory usage: 1.6 KB


In [5]:
climate.shape


(48, 4)

In [4]:
climate


,місто,рік,місяць,температура
0,Львів,2021,1,-3.0
1,Львів,2021,2,-2.0
2,Львів,2021,3,3.1
3,Львів,2021,4,10.0
4,Львів,2021,5,14.3
5,Львів,2021,6,18.7
6,Львів,2021,7,22.1
7,Львів,2021,8,19.7
8,Львів,2021,9,14.0
9,Львів,2021,10,9.0


In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)

base_temp = 8.5
amplitude = 12
city = "Львів"

rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)

        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)
